# 102 · Framework: Patterns of Scientific Logic

In the previous tutorial, we manually interacted with the `Ledger`. While powerful, manual event management becomes complex as your scientific logic grows. 

The `EarlySign` **Framework layer** introduces an architecture that provides core abstractions to manage the lifecycle and lineage of an analysis:

1. **Projectors**: Pure functional views that translate history into scientific objects.
2. **Sessions & Writers**: Orchestrators that handle reading, writing, and lineage tracking.
3. **Entities**: Identifiable aggregates that support efficient snapshotting.
4. **Trace & Lineage**: The statistical "provenance" that links results to raw data.

In this tutorial, we will explore these components by building a simple "Counter" system.

## 1. Setup

We initialize a Ledger as before.

In [1]:
import json
from typing import Optional

import ibis
from pydantic import BaseModel

from earlysign.core.ledger import Ledger
from earlysign.v1.framework.entity.core import Entity
from earlysign.v1.framework.entity.snapshot import Snapshot
from earlysign.v1.framework.projector import ProjectionResult, Projector
from earlysign.v1.framework.session import Session
from earlysign.v1.framework.trace import Traced, TraceId

con = ibis.connect("duckdb://:memory:")
ledger = Ledger(con, "framework_demo").bind(experiment_id="102_demo")
ledger.ensure()

## 2. Pure Functional Views: Projectors

A **`Projector`** is a lens into the ledger. It takes a stream of events and "projects" them into a meaningful object. In `EarlySign`, Projectors are **purely functional**: they don't change the ledger, they only interpret it.

Projectors always return a **`ProjectionResult`** (a subclass of `Traced`), which automatically bundles the resulting data with the specific list of event IDs that generated it.

In [2]:
class CounterState(BaseModel):
    total: int = 0


class IncrementProjector(Projector[CounterState]):
    """Reads ALL increments to get current state."""

    def project(self, table: ibis.Expr) -> ProjectionResult[CounterState]:
        # 1. Select relevant events
        matches = table.filter(table.type == "Increment")
        pdf = matches.execute()

        if pdf.empty:
            return ProjectionResult(data=CounterState(total=0), trace=[])

        # 2. Derive state
        total = (
            pdf["payload"]
            .apply(
                lambda x: json.loads(x)["value"] if isinstance(x, str) else x["value"]
            )
            .sum()
        )

        # 3. Collect lineage (trace)
        trace = [TraceId(str(u)) for u in pdf["uuid"]]

        return ProjectionResult(data=CounterState(total=total), trace=trace)

## 3. Orchestration: Sessions & Writers

The **`Session`** brings components together. It acts as an orchestrator that tracks lineage across multiple reads and writes.

1. **`sess.read(projector)`**: Executes a projector and *merges* its trace into the session's memory.
2. **`sess.commit(fact)`**: Writes a new record to the ledger, automatically tagging it with all traces accumulated during the session.

This creates an **Implicit Web of Proof**: you don't have to manually pass IDs around; the session remembers what influenced your decision.

In [3]:
class Increment(BaseModel):
    value: int


class Summary(BaseModel):
    text: str


# 1. Add some raw data
ledger.insert(Increment(value=10))
ledger.insert(Increment(value=5))

with Session(ledger) as sess:
    # 2. Read using the projector
    # The session now 'knows' we looked at these 2 increments
    result = sess.read(IncrementProjector())
    print(f"Current Total: {result.data.total}")

    # 3. Commit a high-level summary
    # This summary will automatically point back to the increments used for the total
    sess.commit(Summary(text="Initial count completed"))

## 4. Identifiable Aggregates: Entities

While Projectors process history, **`Entity`** provides a way to manage objects with **consistent identity** and **snapshots**. This is essential for efficiency when dealing with millions of events.

An Entity automatically:
1. Finds its latest **Snapshot**.
2. Identifies only the **Delta** (events since the snapshot).
3. Folds the delta into the snapshot using its `compute()` logic.
4. Allows saving a new snapshot via `entity.save(sess, result)`.

In [4]:
class CounterEntity(Entity[CounterState]):
    """An identifiable counter that uses snapshots for efficiency."""

    data_type = CounterState

    @property
    def initial_value(self) -> CounterState:
        return CounterState(total=0)

    def compute(
        self,
        snapshot: Optional[Snapshot[CounterState]],
        delta_expr: ibis.Expr,
        full_table: ibis.Expr,
    ) -> ProjectionResult[CounterState]:
        current_total = snapshot.data.total if snapshot else 0

        # Process only the delta (new increments since snapshot)
        matches = delta_expr.filter(delta_expr.type == "Increment")
        pdf = matches.execute()

        for _, row in pdf.iterrows():
            p = row["payload"]
            val = json.loads(p)["value"] if isinstance(p, str) else p["value"]
            current_total += val

        # The trace combines the snapshot and any new events
        trace = [TraceId(str(u)) for u in pdf["uuid"]]
        if snapshot and snapshot.uuid:
            trace.insert(0, TraceId(str(snapshot.uuid)))

        return ProjectionResult(data=CounterState(total=current_total), trace=trace)

In [5]:
with Session(ledger) as sess:
    # Read using the efficient Entity pattern
    counter = CounterEntity(identity="shared_counter")
    result = sess.read(counter)

    # Save a snapshot to speed up future reads
    counter.save(sess, result)

    print(f"Entity Total: {result.data.total}")

## 5. Scientific Provenance: The Trace

Underlying everything we've done is the **Trace**. In science, a result is only as good as its provenance. `EarlySign` tracks exactly which inputs influenced every computation.

- **`TraceId`**: A type-safe identifier for a record UUID.
- **`Traced[T]`**: A generic container that pairs any data `T` with its `trace` (a list of `TraceId`).

By the time you look at a summary or a plot in a GSD monitoring report, `EarlySign` can tell you every single raw event ID that contributed to that specific pixel on the screen.

In [6]:
# Example: A manually traced value
val = Traced(data=42, trace=[TraceId("abc-123")])
print(f"Data: {val.data}, Provenance: {val.trace}")

Data: 42, Provenance: ['abc-123']


## 6. Summary

- **Projectors**: Pure logic for replaying history.
- **Sessions**: Orchestrators that weave the "web of proof".
- **Entities**: Identifiable aggregates that use snapshots for performance.
- **Trace**: The fundamental unit of scientific causality.

In the next section, we will see how these components power real-world **Group Sequential Designs**.